<a href="https://colab.research.google.com/github/minaGalil/Imperial-Capstone/blob/main/Capstone_Week_7_Calling_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# Week 7 BBO Capstone Script
# Now using 16 data points
# GP + SVM + MLP + CNN-inspired model + hyperparameter tuning
# ============================================================

import numpy as np
import pandas as pd
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score

import torch
import torch.nn as nn
import torch.optim as optim

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# Historical Inputs: Weeks 1–6
# ============================================================

historical_inputs = [
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    [
        [0.751825, 0.811946],
        [0.980817, 0.626715],
        [0.996446, 0.737055, 0.850924],
        [0.404477, 0.413254, 0.303108, 0.434359],
        [0.521278, 0.505138, 0.985473, 0.994545],
        [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
        [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
        [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
    ],
    [
        [0.702630, 0.955980],
        [0.700329, 1.000000],
        [0.977206, 0.330727, 0.000017],
        [0.393932, 0.393230, 0.377559, 0.426151],
        [0.449325, 0.603715, 1.000000, 1.000000],
        [0.941155, 0.654101, 0.079248, 0.961137, 0.229746],
        [0.153100, 0.236737, 0.190018, 0.000000, 0.365128, 0.744155],
        [0.103762, 0.017606, 0.204751, 0.194101, 0.745367, 0.674182, 0.125362, 0.044024],
    ],
    [
        [0.012715, 0.992415],
        [0.026867, 0.999598],
        [0.955194, 0.008179, 0.700062],
        [0.425212, 0.377719, 0.427527, 0.420346],
        [0.479838, 0.890814, 1.000000, 1.000000],
        [0.137088, 0.268975, 0.283122, 0.986752, 0.045377],
        [0.000000, 0.450181, 0.211028, 0.000000, 0.353842, 1.000000],
        [0.000000, 0.000000, 0.301242, 0.216500, 0.000000, 1.000000, 0.000000, 1.000000],
    ],
    [
        [0.012658, 0.001204],
        [0.803770, 0.997320],
        [0.034249, 0.018981, 0.451567],
        [0.485961, 0.345148, 0.465576, 0.429334],
        [0.801922, 1.000000, 1.000000, 1.000000],
        [0.586987, 0.182795, 0.646577, 0.709830, 0.091668],
        [0.251627, 0.008515, 0.005607, 0.024728, 0.411581, 0.615802],
        [0.103911, 0.000000, 0.045943, 0.011461, 0.877426, 0.770540, 0.000000, 0.000000],
    ],
]

historical_outputs = [
    [-7.565331332744927e-18, 0.38412771439907634, -0.04757757182509677, -4.83054096204485, 1257.680268889983, -0.7671825918815833, 1.2394822144938658, 9.5430069620611],
    [-7.565331332744927e-18, 0.5530064492925906, -0.03333218343962805, -4.83054096204485, 1257.680268889983, -0.8072367077314392, 1.2394822144938658, 9.5430069620611],
    [2.495129202697582e-35, 0.06646691679004207, -0.04126707799435369, -0.7158150747637886, 1769.2992166742467, -0.721361811601727, 1.493591747104962, 9.7741312776374],
    [-8.189634555426656e-79, 0.5980717829505922, -0.12802055717541724, 0.2735224699683667, 2027.715300336871, -1.6878746934171067, 1.120167075371798, 9.8898511444579],
    [0.0, 0.008971860858651539, -0.18597539887299502, 0.5994256847352308, 3420.1124772143226, -1.0804901380041434, 0.3185961294770624, 9.199106282308],
    [5.086481560089295e-240, 0.16067534592584287, -0.08745083083260485, -1.4671093305843352, 6043.928918914933, -0.40067561435372423, 0.926482363950363, 9.744890331552],
]

# ============================================================
# Helper Functions
# ============================================================

def pair_exists(X, y, point, output):
    same_x = np.all(np.isclose(X, point.reshape(1, -1), atol=1e-10), axis=1)
    same_y = np.isclose(y, output, atol=1e-10)
    return np.any(same_x & same_y)

def normalize_score(values):
    values = np.array(values)
    std = np.std(values)
    if std < 1e-12:
        return np.zeros_like(values)
    return (values - np.mean(values)) / std

# ============================================================
# CNN-inspired 1D surrogate
# ============================================================

class CNN1DSurrogate(nn.Module):
    def __init__(self, input_dim, channels=16):
        super().__init__()
        self.conv1 = nn.Conv1d(1, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear((channels * 2) * input_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)

def train_cnn(X, y, channels=16, epochs=500, lr=0.01):
    X_tensor = torch.tensor(X, dtype=torch.float32)

    y_mean = np.mean(y)
    y_std = np.std(y) if np.std(y) > 1e-12 else 1.0
    y_scaled = (y - y_mean) / y_std
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    model = CNN1DSurrogate(X.shape[1], channels=channels)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()

    return model, y_mean, y_std

def cnn_predict(model, X, y_mean, y_std):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32)
        pred_scaled = model(X_tensor).numpy()
    return pred_scaled * y_std + y_mean

def cnn_gradient(model, x):
    model.eval()
    x_tensor = torch.tensor(x.reshape(1, -1), dtype=torch.float32, requires_grad=True)
    pred = model(x_tensor)
    pred.backward()
    return x_tensor.grad.detach().numpy().ravel()

def cnn_feature_importance(model, X):
    grads = []
    for x in X:
        grads.append(np.abs(cnn_gradient(model, x)))
    importance = np.mean(grads, axis=0)
    if importance.sum() > 0:
        importance = importance / importance.sum()
    return importance

# ============================================================
# Lightweight Hyperparameter Tuning
# ============================================================

def choose_beta(dim, recent_improved):
    if dim <= 3:
        return 1.2 if recent_improved else 1.8
    elif dim <= 5:
        return 1.7 if recent_improved else 2.3
    else:
        return 2.2 if recent_improved else 2.8

def tune_mlp(X, y):
    configs = [
        {"hidden_layer_sizes": (64, 32), "alpha": 0.0005, "learning_rate_init": 0.001},
        {"hidden_layer_sizes": (128, 64), "alpha": 0.001, "learning_rate_init": 0.001},
        {"hidden_layer_sizes": (128, 64, 32), "alpha": 0.0005, "learning_rate_init": 0.0007},
    ]

    best_config = configs[0]
    best_score = -np.inf

    loo = LeaveOneOut()

    for cfg in configs:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(
                hidden_layer_sizes=cfg["hidden_layer_sizes"],
                alpha=cfg["alpha"],
                learning_rate_init=cfg["learning_rate_init"],
                activation="relu",
                max_iter=1200,
                random_state=42
            ))
        ])

        try:
            scores = cross_val_score(model, X, y, cv=loo, scoring="neg_mean_squared_error")
            score = np.mean(scores)
        except Exception:
            score = -np.inf

        if score > best_score:
            best_score = score
            best_config = cfg

    return best_config

def tune_svm(X, y):
    threshold = np.quantile(y, 0.70)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) < 2:
        return None, False, threshold

    configs = [
        {"C": 0.5, "gamma": "scale"},
        {"C": 1.0, "gamma": "scale"},
        {"C": 2.0, "gamma": "scale"},
    ]

    best_model = None
    best_score = -np.inf

    loo = LeaveOneOut()

    for cfg in configs:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                kernel="rbf",
                probability=True,
                C=cfg["C"],
                gamma=cfg["gamma"],
                random_state=42
            ))
        ])

        try:
            scores = cross_val_score(model, X, labels, cv=loo, scoring="accuracy")
            score = np.mean(scores)
        except Exception:
            score = -np.inf

        if score > best_score:
            best_score = score
            best_model = model

    best_model.fit(X, labels)
    return best_model, True, threshold

def choose_cnn_hyperparams(dim):
    if dim <= 3:
        return {"channels": 8, "epochs": 450, "lr": 0.01}
    elif dim <= 5:
        return {"channels": 16, "epochs": 600, "lr": 0.008}
    else:
        return {"channels": 24, "epochs": 750, "lr": 0.006}

# ============================================================
# Main Loop
# ============================================================

results = []
importance_rows = []
tuning_rows = []

for func_id in range(1, 9):

    print("\n" + "=" * 60)
    print(f"Function {func_id} - Week 7")
    print("=" * 60)

    X = np.load(f"function{func_id}_inputs.npy")
    y = np.load(f"function{func_id}_outputs.npy")

    for r in range(len(historical_inputs)):
        point = np.array(historical_inputs[r][func_id - 1], dtype=float)
        output = float(historical_outputs[r][func_id - 1])

        if point.shape[0] != X.shape[1]:
            raise ValueError(f"Function {func_id}: dimension mismatch.")

        if not pair_exists(X, y, point, output):
            X = np.vstack([X, point])
            y = np.append(y, output)

    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_input = X[best_idx]
    best_output = y[best_idx]

    # Did the latest Week 6 output improve the historical best?
    before_latest_best = np.max(y[:-1]) if len(y) > 1 else y[0]
    recent_improved = best_output > before_latest_best

    print("Dataset size:", X.shape)
    print("Best output:", best_output)
    print("Best input:", best_input)

    # -----------------------------
    # Hyperparameter tuning decisions
    # -----------------------------
    beta = choose_beta(dim, recent_improved)
    mlp_cfg = tune_mlp(X, y)
    svm, use_svm, svm_threshold = tune_svm(X, y)
    cnn_cfg = choose_cnn_hyperparams(dim)

    print("Chosen beta:", beta)
    print("Chosen MLP config:", mlp_cfg)
    print("Chosen CNN config:", cnn_cfg)

    # -----------------------------
    # GP model
    # -----------------------------
    kernel = (
        ConstantKernel(1.0)
        * RBF(length_scale=np.ones(dim))
        + WhiteKernel(noise_level=1e-5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-6,
        random_state=42
    )
    gp.fit(X, y)

    # -----------------------------
    # Tuned MLP model
    # -----------------------------
    mlp = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=mlp_cfg["hidden_layer_sizes"],
            activation="relu",
            alpha=mlp_cfg["alpha"],
            learning_rate_init=mlp_cfg["learning_rate_init"],
            max_iter=1200,
            random_state=42
        ))
    ])
    mlp.fit(X, y)

    # -----------------------------
    # Tuned CNN-inspired model
    # -----------------------------
    cnn_model, y_mean, y_std = train_cnn(
        X,
        y,
        channels=cnn_cfg["channels"],
        epochs=cnn_cfg["epochs"],
        lr=cnn_cfg["lr"]
    )

    cnn_importance = cnn_feature_importance(cnn_model, X)

    imp_row = {"Function": func_id}
    for j in range(dim):
        imp_row[f"x{j+1}_importance"] = cnn_importance[j]
    importance_rows.append(imp_row)

    # -----------------------------
    # Candidate generation
    # -----------------------------
    if dim <= 3:
        n_global = 8000
        n_local = 2500
        noise = 0.03
        grad_lr = 0.035
        grad_steps = 25
    elif dim <= 5:
        n_global = 11000
        n_local = 3000
        noise = 0.05
        grad_lr = 0.045
        grad_steps = 30
    else:
        n_global = 16000
        n_local = 3500
        noise = 0.08
        grad_lr = 0.055
        grad_steps = 35

    global_candidates = np.random.uniform(0, 1, size=(n_global, dim))
    local_candidates = np.clip(
        best_input + np.random.normal(0, noise, size=(n_local, dim)),
        0,
        1
    )

    # CNN gradient candidates from top 5 points
    top_k = min(5, len(y))
    top_indices = np.argsort(y)[-top_k:]
    grad_candidates = []

    for idx in top_indices:
        x = X[idx].copy()

        for _ in range(grad_steps):
            grad = cnn_gradient(cnn_model, x)
            grad = grad * (0.5 + cnn_importance)

            norm = np.linalg.norm(grad)
            if norm > 1e-12:
                grad = grad / norm

            x = np.clip(x + grad_lr * grad, 0, 1)

        grad_candidates.append(x)

    grad_candidates = np.array(grad_candidates)

    candidates = np.vstack([
        global_candidates,
        local_candidates,
        grad_candidates
    ])

    # -----------------------------
    # Model scoring
    # -----------------------------
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    mlp_pred = mlp.predict(candidates)
    cnn_pred = cnn_predict(cnn_model, candidates, y_mean, y_std)

    if use_svm:
        svm_prob = svm.predict_proba(candidates)[:, 1]
    else:
        svm_prob = np.ones(len(candidates))

    grad_strength = np.array([
        np.linalg.norm(cnn_gradient(cnn_model, c))
        for c in candidates
    ])

    final_score = (
        0.30 * normalize_score(ucb)
        + 0.25 * normalize_score(mlp_pred)
        + 0.25 * normalize_score(cnn_pred)
        + 0.10 * svm_prob
        + 0.10 * normalize_score(grad_strength)
    )

    best_candidate_idx = np.argmax(final_score)
    new_point = np.clip(candidates[best_candidate_idx], 0, 1)

    query = "-".join([f"{v:.6f}" for v in new_point])

    print("Week 7 Submission:", query)

    results.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Week7_Query": query,
        "Beta": beta,
        "MLP_Hidden_Layers": str(mlp_cfg["hidden_layer_sizes"]),
        "MLP_Alpha": mlp_cfg["alpha"],
        "MLP_Learning_Rate": mlp_cfg["learning_rate_init"],
        "CNN_Channels": cnn_cfg["channels"],
        "CNN_Epochs": cnn_cfg["epochs"],
        "CNN_LR": cnn_cfg["lr"]
    })

# ============================================================
# Save results
# ============================================================

pd.DataFrame(results).to_csv("Week7_Submissions.csv", index=False)
pd.DataFrame(importance_rows).to_csv("Week7_CNN_Feature_Importance.csv", index=False)

print("\nDone ✅")
print("Saved: Week7_Submissions.csv")
print("Saved: Week7_CNN_Feature_Importance.csv")


Function 1 - Week 7
Dataset size: (15, 2)
Best output: 7.710875114502849e-16
Best input: [0.73102363 0.73299988]
Chosen beta: 1.8
Chosen MLP config: {'hidden_layer_sizes': (128, 64, 32), 'alpha': 0.0005, 'learning_rate_init': 0.0007}
Chosen CNN config: {'channels': 8, 'epochs': 450, 'lr': 0.01}
Week 7 Submission: 0.058297-0.998421

Function 2 - Week 7
Dataset size: (16, 2)
Best output: 0.6112052157614438
Best input: [0.70263656 0.9265642 ]
Chosen beta: 1.8
Chosen MLP config: {'hidden_layer_sizes': (128, 64, 32), 'alpha': 0.0005, 'learning_rate_init': 0.0007}
Chosen CNN config: {'channels': 8, 'epochs': 450, 'lr': 0.01}
Week 7 Submission: 0.854811-0.022906

Function 3 - Week 7
Dataset size: (21, 3)
Best output: -0.03333218343962805
Best input: [0.522581 0.591593 0.350176]
Chosen beta: 1.8
Chosen MLP config: {'hidden_layer_sizes': (64, 32), 'alpha': 0.0005, 'learning_rate_init': 0.001}
Chosen CNN config: {'channels': 8, 'epochs': 450, 'lr': 0.01}
Week 7 Submission: 0.507856-0.945814-0.0